In [1]:
#1
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#9
!pip uninstall -y peft
!pip install peft==0.10.0

Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
  Using cached peft-0.10.0-py3-none-any.whl.metadata (13 kB)
Using cached peft-0.10.0-py3-none-any.whl (199 kB)


In [3]:
#3
!pip install -U transformers datasets peft accelerate evaluate

  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
Using cached peft-0.19.1-py3-none-any.whl (680 kB)
  Attempting uninstall: peft
    Found existing installation: peft 0.10.0
    Uninstalling peft-0.10.0:
      Successfully uninstalled peft-0.10.0


In [4]:
#4
import transformers
import peft

In [5]:
#5
import os
os.listdir()

['.config', 'dogri-datasetv01.csv', 'drive', 'sample_data']

In [6]:
#6
!pip install -U transformers datasets peft accelerate sacrebleu evaluate

In [7]:
#7
import pandas as pd
df = pd.read_csv("dogri-datasetv01.csv")
df.head()

,src_dogri,tgt_english,category
0,मेरे कोल आ,come to me,general
1,जा,go,general
2,इद्दर बेई जा,sit here,general
3,खलोई आ,stand up,general
4,बलगी जा,wait,general


In [8]:
#8
from datasets import load_dataset

dataset = load_dataset("csv", data_files="dogri-datasetv01.csv")
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

In [9]:
#10
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer.src_lang = "doi_Deva"
tokenizer.tgt_lang = "eng_Latn"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [10]:
#11
def preprocess(example):
    inputs = tokenizer(
        example["src_dogri"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    targets = tokenizer(
        example["tgt_english"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

dataset = dataset.map(preprocess)

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [13]:
#12
!pip install --upgrade torchao
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [14]:
#13
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=30,
    save_strategy="no",
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,14.321552
20,14.224414
30,14.019504
40,13.851735
50,13.626773
60,13.496707
70,13.320160
80,13.160767
90,13.050804
100,12.946324


TrainOutput(global_step=300, training_loss=12.782770360310872, metrics={'train_runtime': 7838.8207, 'train_samples_per_second': 0.153, 'train_steps_per_second': 0.038, 'total_flos': 326152853913600.0, 'train_loss': 12.782770360310872, 'epoch': 30.0})

In [15]:
#14
def translate(text):
    # This is the "no-fail" way to get the language ID
    target_lang = "eng_Latn"
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_lang)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_new_tokens=64
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [16]:
#15
predictions = []
references = []
dogri_sentences = []

for example in dataset["test"]:
    pred = translate(example["src_dogri"])
    predictions.append(pred)
    references.append(example["tgt_english"])
    dogri_sentences.append(example["src_dogri"])

Both `max_new_tokens` (=64) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

In [17]:
#16
import evaluate

# BLEU (baseline)
bleu = evaluate.load("bleu")
bleu_score = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

print("BLEU:", bleu_score["bleu"])


# METEOR (MAIN METRIC)
meteor = evaluate.load("meteor")
meteor_score = meteor.compute(
    predictions=predictions,
    references=references
)

print("METEOR:", meteor_score["meteor"])

BLEU: 0.0


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


METEOR: 0.2228830992019632


In [18]:
#17
import pandas as pd

df = pd.DataFrame({
    "dogri": dogri_sentences,
    "prediction": predictions,
    "reference": references
})

# Save it
df.to_csv("dogri_results_final.csv", index=False)
print("File saved! Check the folder icon on the left.")

File saved! Check the folder icon on the left.


In [19]:
#18
# Check the first pair
print(f"Pred: {predictions[0]}")
print(f"Ref: {references[0]}")

# Check types - References should usually be a list of lists
print(f"Type Pred: {type(predictions[0])}")
print(f"Type Ref: {type(references[0])}")

Pred: Listen to my whisper.
Ref: listen to me
Type Pred: <class 'str'>
Type Ref: <class 'str'>


In [20]:
import json

scores = {
    "bleu": bleu_score,
    "meteor": meteor_score
}

with open("/content/drive/MyDrive/scores.json", "w") as f:
    json.dump(scores, f)

In [22]:
model.save_pretrained("/content/drive/MyDrive/Dogri_translation")
tokenizer.save_pretrained("/content/drive/MyDrive/Dogri_translation")

('/content/drive/MyDrive/Dogri_translation/tokenizer_config.json',
 '/content/drive/MyDrive/Dogri_translation/tokenizer.json')